In [1]:
import numpy as np
import librosa
import noisereduce as nr
from scipy import signal
import joblib


In [2]:
model = joblib.load("models/cough_detector.pkl")
scaler = joblib.load("models/scaler.pkl")
label_encoder = joblib.load("models/label_encoder.pkl")

print("Model, scaler, and label encoder loaded successfully!")


Model, scaler, and label encoder loaded successfully!


In [3]:
def extract_features(file_path, sr=22050):
    try:
        audio, sr = librosa.load(file_path, sr=sr, duration=5)
        audio = nr.reduce_noise(y=audio, sr=sr)
        audio, _ = librosa.effects.trim(audio, top_db=20)

        # SAFETY: skip files that are too short
        if len(audio) < 2000:  
            return None

        features = []

        # MFCC (40 coeffs)
        mfccs = librosa.feature.mfcc(y=audio, sr=sr, n_mfcc=40)
        features.extend(np.mean(mfccs, axis=1))
        features.extend(np.std(mfccs, axis=1))
        features.extend(np.max(mfccs, axis=1))

        # Delta MFCC
        delta = librosa.feature.delta(mfccs)
        features.extend(np.mean(delta, axis=1))
        features.extend(np.std(delta, axis=1))

        # Spectral features
        features.extend([
            np.mean(librosa.feature.spectral_centroid(y=audio, sr=sr)),
            np.mean(librosa.feature.spectral_rolloff(y=audio, sr=sr)),
            np.mean(librosa.feature.zero_crossing_rate(audio)),
            np.mean(librosa.feature.rms(y=audio))
        ])
        # Cough energy band (500–2000 Hz)
        sos = signal.butter(4, [500, 2000], btype='band', fs=sr, output='sos')
        filtered = signal.sosfilt(sos, audio)
        features.append(np.sum(filtered**2) / (np.sum(audio**2) + 1e-8))

        return np.array(features)

    except Exception as e:
        # print("Error:", e)   # optional debug
        return None

In [4]:
def extract_features_from_chunk(audio, sr=22050):
    try:
        if len(audio) < 500:
            return None

        features = []

        mfccs = librosa.feature.mfcc(y=audio, sr=sr, n_mfcc=40)
        features.extend(np.mean(mfccs, axis=1))
        features.extend(np.std(mfccs, axis=1))
        features.extend(np.max(mfccs, axis=1))

        delta = librosa.feature.delta(mfccs)
        features.extend(np.mean(delta, axis=1))
        features.extend(np.std(delta, axis=1))

        features.extend([
            np.mean(librosa.feature.spectral_centroid(y=audio, sr=sr)),
            np.mean(librosa.feature.spectral_rolloff(y=audio, sr=sr)),
            np.mean(librosa.feature.zero_crossing_rate(audio)),
            np.mean(librosa.feature.rms(y=audio))
        ])

        sos = signal.butter(4, [500, 2000], btype='band', fs=sr, output='sos')
        filtered = signal.sosfilt(sos, audio)
        features.append(np.sum(filtered**2) / (np.sum(audio**2) + 1e-8))

        return np.array(features)
    except:
        return None


In [5]:
def predict_audio(file_path):
    feat = extract_features(file_path)

    if feat is None:
        return "Error: Invalid or too short audio."

    feat = feat.reshape(1, -1)
    feat_s = scaler.transform(feat)

    pred = model.predict(feat_s)[0]   # use model instead of gb

    return "COUGH" if pred == 1 else "NON-COUGH"


In [6]:
def predict_audio_sliding(file_path, sr=22050, win_size=1.0, step=0.5):
    try:
        audio, sr = librosa.load(file_path, sr=sr)
        audio = nr.reduce_noise(y=audio, sr=sr)

        window_samples = int(win_size * sr)
        step_samples = int(step * sr)

        cough_votes = 0
        total_windows = 0

        for start in range(0, len(audio) - window_samples, step_samples):
            chunk = audio[start:start + window_samples]

            mf = extract_features_from_chunk(chunk, sr)
            if mf is None:
                continue

            mf = scaler.transform([mf])
            pred = model.predict(mf)[0]

            total_windows += 1
            if pred == 1:
                cough_votes += 1

        if cough_votes > 0:
            return "COUGH (detected in some segment)"
        else:
            return "NON-COUGH"
    except Exception as e:
        return f"Error: {e}"


In [7]:
file_path = r"C:\Users\Anup thakur\Downloads\WhatsApp Audio 2025-12-12 at 11.19.09_c6f1c21f.waptt.wav"
print("Prediction:", predict_audio(file_path))


Prediction: NON-COUGH
